In [1]:
import os
import pandas
import xarray
import hsclient
import numpy as np
import earthaccess
import matplotlib.pyplot as plt
import datetime

# Replace xxxx with your username and password, or import them from a separate file
os.environ["EARTHDATA_USERNAME"] = "xxxx"
os.environ["EARTHDATA_PASSWORD"] = "xxxx"

# Login to EarthAccess
auth = earthaccess.login(strategy="environment")

# Search and locate granules
granule_info = earthaccess.search_data(
    short_name="SWOT_L4_DAWG_SOS_DISCHARGE",
    temporal=("2023-01-01", "2025-01-01"),
)

# Display available granules
for i, collection in enumerate(granule_info):
    print(f'Collection Index: {i}')
    print(collection['umm']['CollectionReference']['ShortName'])
    print(collection['umm']['DataGranule']['ArchiveAndDistributionInformation'][-1]['Name'])
    print(collection['umm']['SpatialExtent']['HorizontalSpatialDomain']['Geometry']['BoundingRectangles'])
    print()

Collection Index: 0
SWOT_L4_DAWG_SOS_DISCHARGE
na_sword_v16_SOS_unconstrained_0001_20240611T010141_results.nc
[{'WestBoundingCoordinate': -166.397, 'SouthBoundingCoordinate': 8.09, 'EastBoundingCoordinate': 8.09, 'NorthBoundingCoordinate': 82.311}]

Collection Index: 1
SWOT_L4_DAWG_SOS_DISCHARGE
na_sword_v16_SOS_unconstrained_0001_20240726T123358_results.nc
[{'WestBoundingCoordinate': -166.397, 'SouthBoundingCoordinate': 8.09, 'EastBoundingCoordinate': 8.09, 'NorthBoundingCoordinate': 82.311}]

Collection Index: 2
SWOT_L4_DAWG_SOS_DISCHARGE
eu_sword_v16_SOS_unconstrained_0001_20240726T123345_results.nc
[{'WestBoundingCoordinate': -21.794, 'SouthBoundingCoordinate': 25.382, 'EastBoundingCoordinate': 25.382, 'NorthBoundingCoordinate': 81.115}]

Collection Index: 3
SWOT_L4_DAWG_SOS_DISCHARGE
sa_sword_v16_SOS_unconstrained_0001_20240726T123343_priors.nc
[{'WestBoundingCoordinate': -81.139, 'SouthBoundingCoordinate': -52, 'EastBoundingCoordinate': -52, 'NorthBoundingCoordinate': 11.097}]

C

In [2]:
files = earthaccess.open(granule_info)
files

QUEUEING TASKS | :   0%|          | 0/24 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/24 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/24 [00:00<?, ?it/s]

[<File-like object HTTPFileSystem, https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/SWOT_L4_DAWG_SOS_DISCHARGE/na_sword_v16_SOS_unconstrained_0001_20240611T010141_results.nc>,
 <File-like object HTTPFileSystem, https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/SWOT_L4_DAWG_SOS_DISCHARGE/na_sword_v16_SOS_unconstrained_0001_20240611T010141_priors.nc>,
 <File-like object HTTPFileSystem, https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/SWOT_L4_DAWG_SOS_DISCHARGE/na_sword_v16_SOS_unconstrained_0001_20240726T123358_results.nc>,
 <File-like object HTTPFileSystem, https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/SWOT_L4_DAWG_SOS_DISCHARGE/na_sword_v16_SOS_unconstrained_0001_20240726T123358_priors.nc>,
 <File-like object HTTPFileSystem, https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/SWOT_L4_DAWG_SOS_DISCHARGE/eu_sword_v16_SOS_unconstrained_0001_20240726T123345_results.nc>,
 <File-like 

In [3]:
%%time
print(f'Loading the "Reaches" group in file: {files[4].full_name}')

# Change the file number acordingly if you can working on a non-North American river
ds_reaches = xarray.open_dataset(files[2],
                                 group='reaches',
                                 engine='h5netcdf',
                                 decode_cf=False,    
                                 decode_times=False, 
                                 decode_coords=False)
ds_reaches

Loading the "Reaches" group in file: https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/SWOT_L4_DAWG_SOS_DISCHARGE/eu_sword_v16_SOS_unconstrained_0001_20240726T123345_results.nc
CPU times: total: 1.66 s
Wall time: 1min 10s


<xarray.Dataset> Size: 12MB
Dimensions:       (num_reaches: 38048)
Dimensions without coordinates: num_reaches
Data variables:
    reach_id      (num_reaches) int64 304kB ...
    x             (num_reaches) float64 304kB ...
    y             (num_reaches) float64 304kB ...
    river_name    (num_reaches) <U72 11MB ...
    observations  (num_reaches) object 304kB ...
    time          (num_reaches) object 304kB ...

In [4]:
%%time
print(f'Loading the "hivdi" group in file: {files[4].full_name}')

# Change the file number acordingly if you can working on a non-North American river
ds_hivdi = xarray.open_dataset(files[2],
                           group='hivdi',
                           engine='h5netcdf',
                           decode_cf=False,    
                           decode_times=False, 
                           decode_coords=False,
                        )
ds_hivdi

Loading the "hivdi" group in file: https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/SWOT_L4_DAWG_SOS_DISCHARGE/eu_sword_v16_SOS_unconstrained_0001_20240726T123345_results.nc
CPU times: total: 1 s
Wall time: 49.1 s


<xarray.Dataset> Size: 1MB
Dimensions:  (num_reaches: 38048)
Dimensions without coordinates: num_reaches
Data variables:
    Q        (num_reaches) object 304kB ...
    A0       (num_reaches) float64 304kB ...
    beta     (num_reaches) float64 304kB ...
    alpha    (num_reaches) float64 304kB ...

In [5]:
from ipywidgets import interact, widgets
import folium

# User-defined Reach ID Input
def plot_reach(reach_id):
    try:
        reach_id = int(reach_id)
        reach_index = np.where(ds_reaches['reach_id'].values == reach_id)[0]
        if len(reach_index) == 0:
            print(f"Reach ID {reach_id} not found.")
            return

        reach_index = reach_index[0]

        # Extract location and discharge
        lat = ds_reaches['y'][reach_index].values
        lon = ds_reaches['x'][reach_index].values
        discharge = ds_hivdi['Q'][reach_index].values

        # Display reach location
        print(f"Reach ID: {reach_id}")
        print(f"Location: Latitude {lat:.4f}, Longitude {lon:.4f}")

        # Create interactive map with Folium
        m = folium.Map(location=[lat, lon], zoom_start=10)
        folium.Marker(
            location=[lat, lon],
            popup=f"Reach ID: {reach_id}\nLat: {lat:.4f}, Lon: {lon:.4f}",
            icon=folium.Icon(color='red', icon='info-sign')
        ).add_to(m)

        # Display map
        display(m)

        # Plot discharge time series
        times = ds_reaches['time'][reach_index].values
        base_time = datetime.datetime(2000, 1, 1)
        dates = [base_time + datetime.timedelta(seconds=int(t)) for t in times]

        plt.figure(figsize=(10, 5))
        plt.plot(dates, discharge, marker='o', color='b')
        plt.title(f"Discharge Time Series for Reach {reach_id}")
        plt.xlabel("Date")
        plt.ylabel("Discharge (m³/s)")
        plt.grid(True)
        plt.show()

    except ValueError:
        print("Please enter a valid numerical Reach ID.")

# Interactive input for user-defined reach ID
interact(plot_reach, reach_id=widgets.Text(value='', placeholder='Enter Reach ID'))

interactive(children=(Text(value='', description='reach_id', placeholder='Enter Reach ID'), Output()), _dom_cl…

<function __main__.plot_reach(reach_id)>

In [6]:
# Make sure to change the file path
csv_path = 'xxxx'
df_measured = pandas.read_csv(csv_path)

# Format date
df_measured['date'] = df_measured["Time_('dd-mm-yyyy')"].str.replace("'", "", regex=False)
df_measured['date'] = pandas.to_datetime(df_measured['date'], format='%d-%m-%Y')
df_measured = df_measured.rename(columns={'Reach_ID': 'reach_id', 'Q_(m^3/s_daily)': 'discharge'})

def compare_discharge(reach_id):
    try:
        reach_id = int(reach_id)
        reach_index = np.where(ds_reaches['reach_id'].values == reach_id)[0]
        if len(reach_index) == 0:
            print(f"Reach ID {reach_id} not found.")
            return

        reach_index = reach_index[0]

        # Extract estimated discharge date
        discharge_algo_q = ds_hivdi['Q'][reach_index].values
        time_raw = ds_reaches['time'][reach_index].values
        base_time = datetime.datetime(2000, 1, 1)
        time_str = [(base_time + datetime.timedelta(seconds=int(t))).strftime("%Y-%m-%dT%H:%M:%S") for t in time_raw]
        valid_times_dt = [datetime.datetime.strptime(t, "%Y-%m-%dT%H:%M:%S") for t in time_str if t != "NO_DATA"]
        estimated_discharge_dict = {dt.date(): discharge_algo_q[i] for i, dt in enumerate(valid_times_dt)}

        # Filter measured discharge by reach ID
        df_reach = df_measured[df_measured['reach_id'] == reach_id]

        matching_dates = []
        measured_values = []
        estimated_values = []

        for _, row in df_reach.iterrows():
            user_date_dt = row['date'].date()
            measured_q = row['discharge']
            if user_date_dt in estimated_discharge_dict:
                matching_dates.append(str(user_date_dt))
                measured_values.append(measured_q)
                estimated_values.append(estimated_discharge_dict[user_date_dt])

        if not matching_dates:
            print("No matching dates found for comparison.")
            return

        # Print comparison table
        print("\nMatching Dates with Measured vs. Estimated Discharge:")
        print(f"{'Date':<12} {'Measured (m³/s)':<18} {'Estimated (m³/s)':<18}")
        print("-" * 50)
        for i in range(len(matching_dates)):
            print(f"{matching_dates[i]:<12} {measured_values[i]:<18.2f} {estimated_values[i]:<18.2f}")

        # Calculate errpr metrics
        measured_values = np.array(measured_values)
        estimated_values = np.array(estimated_values)

        mean_error = np.mean(estimated_values - measured_values)
        rmse = np.sqrt(np.mean((estimated_values - measured_values) ** 2))
        correlation = np.corrcoef(measured_values, estimated_values)[0, 1]

        print("\nError Statistics:")
        print(f"Mean Error: {mean_error:.2f} m³/s")
        print(f"Root Mean Squared Error (RMSE): {rmse:.2f} m³/s")
        print(f"Correlation Coefficient: {correlation:.2f}")

        # Plot estimated vs. measured discharges
        plt.figure(figsize=(10, 6))
        plt.plot(matching_dates, measured_values, 'o-', label="Measured Discharge", color='blue')
        plt.plot(matching_dates, estimated_values, 's--', label="Estimated Discharge", color='red')

        plt.xlabel("Date")
        plt.ylabel("Discharge (m³/s)")
        plt.title(f"Comparison of Measured vs. Estimated Discharge for Reach {reach_id}")
        plt.legend()
        plt.xticks(rotation=90)
        plt.grid(True)
        plt.tight_layout()
        plt.show()

    except ValueError:
        print("Please enter a valid numerical Reach ID.")

# Interactive reach ID input
interact(compare_discharge, reach_id=widgets.Text(value='', placeholder='Enter Reach ID'))

interactive(children=(Text(value='', description='reach_id', placeholder='Enter Reach ID'), Output()), _dom_cl…

<function __main__.compare_discharge(reach_id)>